# 04 · Refined (fato unificado, quarentena e agregados)Aqui acontecem três coisas:1. **Unificação** — yellow e green viram uma tabela só, `fct_taxi_trip`,   com `trip_type` como discriminador e `pickup_datetime`/`dropoff_datetime`   canônicos. Perguntas sobre *toda a frota* deixam de precisar de `UNION ALL`.2. **Qualidade com quarentena** — linhas reprovadas vão para `rej_taxi_trip`   com o motivo, em vez de sumirem num `WHERE`. Dá para auditar e reverter.3. **Agregados** — `agg_trip_monthly` e `agg_trip_hourly`, que respondem as   perguntas do case com uma leitura de poucas linhas.

In [ ]:
%load_ext autoreload%autoreload 2

In [ ]:
# Coloca a raiz do repositório no sys.path.# Sobe diretórios até achar a pasta `src`, de modo que o notebook funcione# tanto em Git Folders quanto em Workspace Files, sem caminho hardcoded.import osimport sys_root = os.getcwd()while _root != "/" and not os.path.isdir(os.path.join(_root, "src")):    _root = os.path.dirname(_root)if _root not in sys.path:    sys.path.insert(0, _root)print("Repo root:", _root)

In [ ]:
from src import configfrom src.processing.trusted_to_refined import TrustedToRefinedProcessorprocessor = TrustedToRefinedProcessor(spark)fato, quarentena, relatorio = processor.process_fact()

## Relatório de qualidadeRegras não bloqueantes aparecem no relatório, mas não retiram a linha do fato.

In [ ]:
display(relatorio.orderBy('blocking', 'rows_failed', ascending=[False, False]))

In [ ]:
%sql-- Quantas linhas foram para a quarentena e por qual motivo.SELECT motivo, COUNT(*) AS linhasFROM ifood_case.refined.rej_taxi_tripLATERAL VIEW explode(_rejection_reasons) AS motivoGROUP BY motivoORDER BY linhas DESC;

## Agregados

In [ ]:
monthly, hourly = processor.process_aggregates()display(monthly)

In [ ]:
display(hourly)

## Otimização física das tabelas de consumo

In [ ]:
%sqlOPTIMIZE ifood_case.refined.fct_taxi_trip ZORDER BY (pickup_datetime, trip_type);ANALYZE TABLE ifood_case.refined.fct_taxi_trip COMPUTE STATISTICS FOR ALL COLUMNS;